# 04b -- Stage 4b: Add DeepDetect-2025 + DeepfakeArt (inpainting)

Broadens the model's AI-generation coverage beyond DFDC's face-swap
deepfakes and the classical-CV Playground manipulations:

- **[DeepDetect-2025](https://www.kaggle.com/datasets/ayushmandatta1/deepdetect-2025)**
  -- ~112k real vs. AI-generated images spanning people, animals,
  nature, urban scenes, and artwork. Fakes come from StyleGAN3,
  DALL-E 3, Midjourney, and Stable Diffusion 3 -- i.e. generic
  text-to-image/GAN generation, not face-swapping. Mapped to
  `type="gan"` for fakes, same convention already used for DFDC.
- **[DeepfakeArt Challenge](https://www.kaggle.com/datasets/danielmao2019/deepfakeart)**
  (inpainting category only) -- WikiArt paintings with a region
  inpainted by Stable Diffusion 2. Each record is a source image,
  an inpainted image, and a **ground-truth mask** of the inpainted
  region. This is the first dataset in the pipeline with *real*
  pixel-level mask supervision for `inpainting` (everything so far --
  DFDC, Playground synthetic data -- has either no mask or a
  classical-CV-generated one), so it directly strengthens the
  segmentation head. The other three DeepfakeArt categories
  (style_transfer, adversarial poisoning, cutmix) don't map cleanly
  onto the current 6-class `manipulation_types` label set, so this
  notebook only pulls in the inpainting category for now.

**Before running**: upload `deepdetect2025.zip` and `deepfakeart.zip`
(exactly as downloaded from Kaggle, not re-zipped) to
`ForensicAI_Lab/data/` on Drive, next to `merged_labels.csv`.

**This notebook is discovery-first, by design**: Kaggle dataset zips
don't publish a guaranteed folder layout, so the extraction cells
below auto-detect real/fake and source/inpaint/mask files by
directory- and filename-keyword matching, then **print what they
found before copying or merging anything**. Read those printouts
before running the merge cell -- if the counts look wrong (e.g. 0
fakes found, or mismatched source/inpaint/mask counts), the dataset's
actual layout differs from what's assumed here and the matching
logic needs a small patch rather than silently training on bad data.

In [ ]:
import sys, os

IS_COLAB = os.path.exists("/var/colab/hostname")
IS_KAGGLE = (not IS_COLAB) and (os.path.exists("/kaggle/input") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ)
PROJECT_ROOT = "/kaggle/working/ForensicAI_Lab" if IS_KAGGLE else "/content/ForensicAI_Lab"

if not os.path.exists(PROJECT_ROOT):
    !git clone https://github.com/MRoy03/Forensic_LAB.git {PROJECT_ROOT}

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from notebooks.utils.colab_common import mount_drive, install_requirements, is_colab

if is_colab():
    mount_drive()
    install_requirements(PROJECT_ROOT)
elif IS_KAGGLE:
    !pip install -q -r requirements-train.txt
else:
    print("Not in Colab/Kaggle -- assuming requirements-train.txt is already installed locally.")

In [ ]:
import os, re, shutil, zipfile
from pathlib import Path
import pandas as pd
import cv2
from config.loader import get_settings, resolve_dataset_root, resolve_path

settings = get_settings()
dataset_root = resolve_dataset_root(settings)
print("Dataset root:", dataset_root)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}


def find_images(root):
    """os.walk(followlinks=True), not Path.glob -- same fix as Stage 2's
    DFDC extraction: pathlib's recurse_symlinks doesn't reliably follow
    Drive/Kaggle-mounted symlinks, plain os.walk does."""
    found = []
    for dirpath, _, filenames in os.walk(root, followlinks=True):
        for fn in filenames:
            if Path(fn).suffix.lower() in IMAGE_EXTS:
                found.append(Path(dirpath) / fn)
    return found


def extract_zip_once(zip_name, extract_to, marker_name):
    """Extracts to **local disk** (never Drive) -- unpacking tens of
    thousands of small files directly onto Drive's FUSE mount is the
    same slow-and-fragile trap Stage 2/3 already hit with DFDC."""
    marker = extract_to / marker_name
    if marker.exists():
        print(f"{extract_to} already extracted -- skipping.")
        return
    zip_path = dataset_root / zip_name
    if not zip_path.exists():
        print(f"[SKIP] {zip_path} not found -- upload it to {dataset_root} to include this dataset.")
        return
    extract_to.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_to)
    marker.write_text("done")
    print(f"Extracted {zip_path} -> {extract_to}")

## DeepDetect-2025: extract + auto-discover real/fake

Assumes real/fake are indicated by a directory literally named
`real`/`fake` (case-insensitive) somewhere in each file's path --
true for essentially every Kaggle deepfake-detection set structured
this way (`train/real`, `train/fake`, `test/real`, `test/fake`, or a
variant). `train` vs. `test` is ignored; Stage 3's own
train/val split already handles that.

In [ ]:
local_root = Path("/content/extra_datasets") if IS_COLAB else Path("/tmp/extra_datasets")
deepdetect_dir = local_root / "deepdetect2025"
extract_zip_once("deepdetect2025.zip", deepdetect_dir, ".extracted")

In [ ]:
def classify_real_fake(path):
    parts = [p.lower() for p in path.parts]
    if "real" in parts:
        return "real"
    if "fake" in parts:
        return "fake"
    parent = path.parent.name.lower()
    if "real" in parent:
        return "real"
    if "fake" in parent:
        return "fake"
    return None


deepdetect_rows = []
deepdetect_unmatched = 0
if deepdetect_dir.exists():
    for p in find_images(deepdetect_dir):
        label = classify_real_fake(p)
        if label is None:
            deepdetect_unmatched += 1
            continue
        manip_type = "none" if label == "real" else "gan"
        deepdetect_rows.append({"filename": str(p), "label": label, "type": manip_type})

print(f"DeepDetect-2025: {len(deepdetect_rows)} matched, {deepdetect_unmatched} unmatched")
if deepdetect_rows:
    print(pd.DataFrame(deepdetect_rows)["label"].value_counts())
else:
    print("No matches -- if the zip IS present and extracted, the real/fake "
          "directory-naming assumption above is wrong for this copy of the "
          "dataset. Run `find_images(deepdetect_dir)[:20]` in a scratch cell "
          "to see actual paths and report back.")

## DeepfakeArt (inpainting category only): extract + auto-discover triplets

Restricts the search to whichever subtree has `inpaint` in an
ancestor directory name (the dataset's other categories --
style_transfer, adv_image, cutmix -- are skipped). Within that
subtree, classifies each file's role (`source` / `inpaint` / `mask`)
by filename keyword, then groups files into records by stripping the
role keyword from the filename stem -- e.g. `0001_source.jpg` and
`0001_mask.jpg` both reduce to group key `0001`.

In [ ]:
deepfakeart_dir = local_root / "deepfakeart"
extract_zip_once("deepfakeart.zip", deepfakeart_dir, ".extracted")

In [ ]:
ROLE_PATTERN = re.compile(r"[_\-]?(source|orig|inpaint(?:ed|ing)?|mask(?:ed)?)[_\-]?", re.IGNORECASE)


def classify_role(path):
    text = (path.stem + " " + path.parent.name).lower()
    if "mask" in text:
        return "mask"
    if "source" in text or "orig" in text:
        return "source"
    if "inpaint" in text:
        return "inpaint"
    return None


def group_key(path):
    return ROLE_PATTERN.sub("", path.stem).strip("_- ") or path.stem


inpainting_files = []
if deepfakeart_dir.exists():
    for p in find_images(deepfakeart_dir):
        if "inpaint" in str(p.parent).lower():
            inpainting_files.append(p)

records = {}
for p in inpainting_files:
    role = classify_role(p)
    if role is None:
        continue
    records.setdefault(group_key(p), {})[role] = p

complete = {k: v for k, v in records.items() if "inpaint" in v and "mask" in v}
print(f"DeepfakeArt inpainting: {len(inpainting_files)} candidate files, "
      f"{len(records)} groups, {len(complete)} with both inpaint+mask")
for k in list(complete)[:5]:
    print(" ", k, {role: str(path.name) for role, path in complete[k].items()})
if not complete:
    print("No complete groups -- if deepfakeart.zip IS present and extracted, "
          "the role/grouping assumptions above don't match this copy of the "
          "dataset. Run `inpainting_files[:20]` in a scratch cell to see actual "
          "paths and report back.")

## Copy matched DeepfakeArt files into place

Renames each matched record to a unique, collision-free filename and
copies the inpainted image into the training images pool and its mask
into `masks_dir` **under the same filename** -- `ForensicImageDataset`
looks up `masks_dir / <image filename>`, so the names must match
exactly. The paired source image (when present) becomes an ordinary
`real`/`none` sample.

In [ ]:
extra_images_dir = local_root / "prepared_images"
extra_masks_dir = local_root / "prepared_masks"
extra_images_dir.mkdir(parents=True, exist_ok=True)
extra_masks_dir.mkdir(parents=True, exist_ok=True)

deepfakeart_rows = []
for i, (key, roles) in enumerate(complete.items()):
    inpaint_name = f"deepfakeart_inpaint_{i:06d}.jpg"
    mask_name = inpaint_name  # must match the image filename exactly

    inpaint_img = cv2.imread(str(roles["inpaint"]))
    mask_img = cv2.imread(str(roles["mask"]), cv2.IMREAD_GRAYSCALE)
    if inpaint_img is None or mask_img is None:
        continue
    cv2.imwrite(str(extra_images_dir / inpaint_name), inpaint_img)
    cv2.imwrite(str(extra_masks_dir / mask_name), mask_img)
    deepfakeart_rows.append({
        "filename": str(extra_images_dir / inpaint_name), "label": "fake", "type": "inpainting",
    })

    if "source" in roles:
        source_name = f"deepfakeart_source_{i:06d}.jpg"
        source_img = cv2.imread(str(roles["source"]))
        if source_img is not None:
            cv2.imwrite(str(extra_images_dir / source_name), source_img)
            deepfakeart_rows.append({
                "filename": str(extra_images_dir / source_name), "label": "real", "type": "none",
            })

print(f"Prepared {len(deepfakeart_rows)} DeepfakeArt rows "
      f"({sum(1 for r in deepfakeart_rows if r['label'] == 'fake')} fake w/ mask, "
      f"{sum(1 for r in deepfakeart_rows if r['label'] == 'real')} real)")

## Persist DeepfakeArt's prepared images/masks to Drive

DeepfakeArt's matched-triplet count is small enough (hundreds to a
few thousand) to safely `copytree` to Drive. **DeepDetect-2025's
~112k matched files are deliberately NOT persisted here** -- writing
that many individual small files straight onto Drive's FUSE mount is
the same slow/fragile trap Stage 2 already hit with DFDC (a run once
stalled for 2+ hours doing exactly this). Its rows stay pointed at
this session's local disk instead; see the warning printed by the
merge cell below.

In [ ]:
if IS_COLAB:
    drive_images_dir = dataset_root / settings.data.images_dir
    drive_masks_dir = dataset_root / settings.data.masks_dir
    drive_images_dir.mkdir(parents=True, exist_ok=True)
    drive_masks_dir.mkdir(parents=True, exist_ok=True)
    if extra_images_dir.exists():
        shutil.copytree(extra_images_dir, drive_images_dir, dirs_exist_ok=True)
    if extra_masks_dir.exists():
        shutil.copytree(extra_masks_dir, drive_masks_dir, dirs_exist_ok=True)
    print("Copied prepared images/masks to Drive.")
else:
    print("Not on Colab -- skipping Drive copy.")

## Merge into the training manifest

**Check the printed counts above first.** Folds whatever
DeepDetect-2025/DeepfakeArt rows were discovered into Stage 3's
`merged_labels.csv` directly (either or both may be empty if a zip
wasn't uploaded yet -- that's fine, it just merges what's available)
-- the same shared, growing manifest Stage 4 also builds onto, so a
later retrain picks up everything with no separate "v2" file to
remember to point at. Re-running this notebook replaces its previous
rows instead of duplicating them. DeepfakeArt rows are rewritten to
the Drive-persisted path from the cell above; DeepDetect-2025 rows
keep their local-session path (see the warning printed below).

In [ ]:
merged_path = dataset_root / "merged_labels.csv"
base_df = pd.read_csv(merged_path) if merged_path.exists() else pd.DataFrame(columns=["filename", "label", "type"])

drive_images_dir = dataset_root / settings.data.images_dir
deepfakeart_rows_persisted = [
    {**row, "filename": str(drive_images_dir / Path(row["filename"]).name)}
    for row in deepfakeart_rows
]
extra_df = pd.DataFrame(deepdetect_rows + deepfakeart_rows_persisted, columns=["filename", "label", "type"])


def _is_ours(filename):
    # deepdetect rows keep their local dir in the path; deepfakeart rows
    # get mixed into drive_images_dir alongside other images, so they're
    # identified by our own generated filename prefix instead.
    return str(deepdetect_dir) in filename or Path(filename).name.startswith("deepfakeart_")


# Idempotent: drop this notebook's previously-merged rows before
# re-adding, so re-running replaces rather than duplicates them.
if not base_df.empty:
    base_df = base_df[~base_df["filename"].apply(_is_ours)]

merged = pd.concat([base_df, extra_df], ignore_index=True)
merged.to_csv(merged_path, index=False)
print(f"merged_labels.csv: {len(base_df)} existing rows + {len(extra_df)} new rows "
      f"-> {merged_path} ({len(merged)} rows)")
print(merged["label"].value_counts())

if deepdetect_rows:
    print("\n[NOTE] DeepDetect-2025 rows point at this session's local disk "
          f"({deepdetect_dir}) and will NOT resolve in a future session -- "
          "re-run this notebook's extraction/discovery/merge cells (fast, "
          "just unzip + a directory scan) before reusing merged_labels.csv there.")

## Next step

Rerun Stage 3's retrain cells (now reading the updated
`merged_labels.csv`) from the latest checkpoint -- same training
loop (augmentation on train only, sqrt-weighted oversampling,
per-class + balanced accuracy, best-balanced-accuracy checkpointing),
no path to change by hand.